# QTEMP revised: direct glitch and dropout extraction

This experimental notebook preserves the original QTEMP family. It detects robust sample-derivative outliers and short-time RMS collapses with active audio on both sides. Results are candidates for listening review, not validated diagnoses of packet loss or acquisition failure.

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

def find_root():
    configured = os.environ.get('PAPER1_PIPELINE_ROOT', '').strip()
    candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate.resolve()
    raise RuntimeError('Set PAPER1_PIPELINE_ROOT to the repository root.')

ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from paper1_qc_reviewed.qtemp_revised import (
    ANALYSIS_FEATURES, DEFAULT_PARAMETERS, MEASUREMENT_VERSION,
    extract_qtemp_revised, parameter_frame,
)
FREEZE = ROOT / 'MAIN outputs' / '00_DATA_FREEZE' / 'v1' / 'bamboo_recording_freeze_ledger.csv'
CANONICAL_IDS = ROOT / 'MAIN outputs' / '02_FEATURE_REVIEWED' / '06_family_freezes' / 'additive_interference' / 'qadd-v4.2.0' / 'tables' / 'qadd_v420_recording_features.csv'
OUTPUT = ROOT / 'MAIN outputs' / '03_EXPERIMENTAL' / 'QTEMP_REVISED'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Version:', MEASUREMENT_VERSION)
print('Output:', OUTPUT)

## Frozen parameters and governed recording population

In [ ]:
display(parameter_frame())
ledger = pd.read_csv(FREEZE)
canonical = pd.read_csv(CANONICAL_IDS, usecols=['logical_recording_id']).drop_duplicates()
population = canonical.merge(ledger[['logical_recording_id', 'media_path']], on='logical_recording_id', how='left', validate='one_to_one')
population = population.loc[population['media_path'].notna()].copy()
assert len(population) == 519, f'Expected 519 frozen recordings, found {len(population)}'
display(population[['logical_recording_id', 'media_path']].head())

## Extract revised features and event timestamps

In [ ]:
feature_rows, event_frames, failures = [], [], []
for row in population.itertuples(index=False):
    recording_id = str(row.logical_recording_id)
    path = Path(row.media_path)
    try:
        waveform, sample_rate = sf.read(path, always_2d=False)
        result = extract_qtemp_revised(waveform, sample_rate, logical_recording_id=recording_id)
        feature_rows.append({**result.recording, 'media_path': str(path)})
        if not result.event_ledger.empty:
            events = result.event_ledger.copy()
            events.insert(0, 'logical_recording_id', recording_id)
            events['media_path'] = str(path)
            event_frames.append(events)
    except Exception as exc:
        failures.append({'logical_recording_id': recording_id, 'media_path': str(path), 'error': repr(exc)})

features = pd.DataFrame(feature_rows)
events = pd.concat(event_frames, ignore_index=True) if event_frames else pd.DataFrame()
failure_table = pd.DataFrame(failures)
features.to_csv(OUTPUT / 'qtemp_revised_features.csv', index=False)
events.to_csv(OUTPUT / 'qtemp_revised_event_ledger.csv', index=False)
failure_table.to_csv(OUTPUT / 'qtemp_revised_failures.csv', index=False)
parameter_frame().to_csv(OUTPUT / 'qtemp_revised_parameters.csv', index=False)
print(f'Measured {len(features)} recordings; failures={len(failure_table)}; events={len(events)}')

## Detection summary and candidate ranking

In [ ]:
summary = pd.DataFrame({
    'feature': ANALYSIS_FEATURES,
    'positive_recordings': [(features[name].fillna(0) > 0).sum() for name in ANALYSIS_FEATURES],
    'median': [features[name].median() for name in ANALYSIS_FEATURES],
    'maximum': [features[name].max() for name in ANALYSIS_FEATURES],
})
summary.to_csv(OUTPUT / 'qtemp_revised_summary.csv', index=False)
display(summary)
ranked = features.sort_values(
    ['qtemp_revised_dropout_event_rate_per_min', 'qtemp_revised_glitch_rate_per_min'],
    ascending=False,
)
display(ranked[['logical_recording_id', *ANALYSIS_FEATURES]].head(30))

## Listen and inspect a selected candidate

In [ ]:
RECORDING_ID = ranked.iloc[0]['logical_recording_id']  # change this value to review another row
selected = features.loc[features.logical_recording_id.eq(RECORDING_ID)].iloc[0]
audio, fs = sf.read(selected.media_path, always_2d=False)
mono = audio.mean(axis=1) if audio.ndim == 2 else audio
selected_events = events.loc[events.logical_recording_id.eq(RECORDING_ID)] if not events.empty else events
display(selected_events)
display(Audio(mono, rate=fs))
fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
time = np.arange(len(mono)) / fs
axes[0].plot(time, mono, linewidth=.4)
axes[0].set(title=f'{RECORDING_ID}: waveform', xlabel='Time (s)', ylabel='Amplitude')
axes[1].specgram(mono, Fs=fs, NFFT=1024, noverlap=768, cmap='magma')
axes[1].set(title='Spectrogram', xlabel='Time (s)', ylabel='Frequency (Hz)')
for event in selected_events.itertuples(index=False):
    for axis in axes:
        axis.axvspan(event.start_sec, max(event.end_sec, event.start_sec + .005), alpha=.25, color='cyan')
plt.show()